# Ekstraksi Dataset Abu Vulkanik dari VONA Merapi

Notebook ini membaca `vona_reports_merapi.csv`, mengekstrak fitur yang relevan ke skema `ash_dataset.csv`, lalu menyimpan hasilnya sebagai dataset baru.

Kolom yang diisi dari teks VONA:
- `tinggi_letusan_m`
- `kec_angin_km_jam`
- `arah_angin_deg`
- `amplitudo`
- `max_time`
- `jarak_km` bila ada informasi jarak luncur atau run out distance

Kolom `luas_km2`, `sudut_deg`, dan `radius_km` dibiarkan kosong jika memang tidak tersedia secara eksplisit di data sumber.

In [1]:
from pathlib import Path

import math

import re



import pandas as pd



WORKDIR = Path.cwd()

PROJECT_ROOT = WORKDIR if (WORKDIR / 'data').exists() else WORKDIR.parent

DATA_DIR = PROJECT_ROOT / 'data'

SOURCE_CSV = DATA_DIR / 'vona_reports_filtered_complete_v2.csv'

TEMPLATE_CSV = DATA_DIR / 'ash_dataset.csv'

OUTPUT_CSV = DATA_DIR / 'filtered_semeru.csv'



source_df = pd.read_csv(SOURCE_CSV)

template_columns = [column.strip() for column in pd.read_csv(TEMPLATE_CSV, nrows=0).columns.tolist()]



print(f'Rows sumber: {len(source_df)}')

print(f'Kolom template: {template_columns}')

Rows sumber: 4410
Kolom template: ['tinggi_letusan_m', 'kec_angin_km_jam', 'arah_angin_deg', 'amplitudo', 'max_time', 'jarak_km', 'luas_km2', 'sudut_deg', 'radius_km']


In [2]:
DIRECTION_MAP = {
    'north': 0.0,
    'north northeast': 22.5,
    'northeast': 45.0,
    'north east': 45.0,
    'east northeast': 67.5,
    'east': 90.0,
    'east southeast': 112.5,
    'southeast east': 112.5,
    'southeast': 135.0,
    'south east': 135.0,
    'south southeast': 157.5,
    'south': 180.0,
    'south southwest': 202.5,
    'southwest': 225.0,
    'south west': 225.0,
    'west southwest': 247.5,
    'west': 270.0,
    'west northwest': 292.5,
    'northwest': 315.0,
    'north west': 315.0,
    'north northwest': 337.5,
}

WIND_SPEED_MAP = {
    'weak': 10.0,
    'light': 20.0,
    'moderate': 35.0,
    'fresh': 50.0,
    'strong': 65.0,
}

def is_missing(value):
    return pd.isna(value) or str(value).strip() == ''

def combine_text(row, columns):
    parts = []
    for column in columns:
        value = row.get(column)
        if not is_missing(value):
            parts.append(str(value))
    return ' '.join(parts)

def parse_number(token):
    token = str(token).strip()
    if not token:
        return math.nan

    token = token.replace('−', '-')
    if ',' in token and '.' in token:
        if token.rfind(',') > token.rfind('.'):
            token = token.replace('.', '').replace(',', '.')
        else:
            token = token.replace(',', '')
    elif token.count(',') == 1 and len(token.split(',')[-1]) in {1, 2}:
        token = token.replace(',', '.')
    elif token.count(',') > 1:
        token = token.replace(',', '')

    try:
        return float(token)
    except ValueError:
        return math.nan

def circular_mean(degrees):
    radians = [math.radians(value) for value in degrees]
    sin_total = sum(math.sin(value) for value in radians)
    cos_total = sum(math.cos(value) for value in radians)
    angle = math.degrees(math.atan2(sin_total, cos_total)) % 360
    return angle

def direction_phrase_to_degrees(phrase):
    if not phrase:
        return math.nan

    cleaned = str(phrase).lower().strip(' .,:;()')
    cleaned = cleaned.replace('-', ' ')
    cleaned = re.sub(r'\s+', ' ', cleaned)
    cleaned = re.sub(r'^(the|toward|towards|to|from)\s+', '', cleaned)

    if cleaned in DIRECTION_MAP:
        return DIRECTION_MAP[cleaned]

    if ' to ' in cleaned:
        parts = [part.strip() for part in cleaned.split(' to ') if part.strip()]
        values = [direction_phrase_to_degrees(part) for part in parts]
        values = [value for value in values if not math.isnan(value)]
        if values:
            return circular_mean(values)

    return math.nan

def extract_height_m(row):
    text = combine_text(row, ['Volcanic_Cloud_Height', 'Description', 'Volcanic_Activity_Summary'])

    above_summit = re.search(r'\(([^()]+)\s*M\)\s*above summit', text, flags=re.IGNORECASE)
    if above_summit:
        return parse_number(above_summit.group(1))

    above_sea = re.search(r'\(([^()]+)\s*M\)\s*above sea level', text, flags=re.IGNORECASE)
    if above_sea:
        sea_level_height = parse_number(above_sea.group(1))
        summit_height = parse_number(row.get('Summit_Elevation_M'))
        if not math.isnan(sea_level_height) and not math.isnan(summit_height):
            return max(sea_level_height - summit_height, 0.0)
        return sea_level_height

    return math.nan

def extract_wind_speed_kmh(row):
    text = combine_text(row, ['Remarks', 'Other_Volcanic_Cloud_Info', 'Description'])

    numeric_match = re.search(r'(\d+(?:[.,]\d+)?)\s*(?:km\s*/?\s*h|km\s*/?\s*jam)', text, flags=re.IGNORECASE)
    if numeric_match:
        return parse_number(numeric_match.group(1))

    category_match = re.search(
        r'(weak|light|moderate|fresh|strong)(?:\s+to\s+(weak|light|moderate|fresh|strong))?\s+winds?',
        text,
        flags=re.IGNORECASE,
    )
    if category_match:
        first = WIND_SPEED_MAP[category_match.group(1).lower()]
        second_label = category_match.group(2)
        if second_label:
            second = WIND_SPEED_MAP[second_label.lower()]
            return (first + second) / 2
        return first

    return math.nan

def extract_wind_direction_deg(row):
    texts = [
        row.get('Other_Volcanic_Cloud_Info', ''),
        row.get('Remarks', ''),
        row.get('Description', ''),
    ]

    patterns = [
        (r'wind direction to(?: the)? ([a-z\-\s]+?)(?:[.,;]|$)', 'to'),
        (r'winds? blowing toward(?: the)? ([a-z\-\s]+?)(?:[.,;]|$)', 'to'),
        (r'winds? from(?: the)? ([a-z\-\s]+?)(?:[.,;]|$)', 'from'),
        (r'ash(?:-|\s)?cloud[^.]*?moving from ([a-z\-\s]+?)(?:[.,;]|$)', 'to'),
        (r'ash(?:-|\s)?cloud[^.]*?moving to(?: the)? ([a-z\-\s]+?)(?:[.,;]|$)', 'to'),
        (r'pyroclastic density current heading ([a-z\-\s]+?)(?:[.,;]|$)', 'to'),
        (r'pdc moving to(?: the)? ([a-z\-\s]+?)(?:[.,;]|$)', 'to'),
    ]

    for text in texts:
        if is_missing(text):
            continue

        for pattern, mode in patterns:
            match = re.search(pattern, str(text), flags=re.IGNORECASE)
            if not match:
                continue

            degrees = direction_phrase_to_degrees(match.group(1))
            if math.isnan(degrees):
                continue

            if mode == 'from':
                return (degrees + 180.0) % 360.0
            return degrees

    return math.nan

def extract_amplitude_mm(row):
    text = combine_text(row, ['Remarks'])
    matches = []

    patterns = [
        r'maximum amplitud(?:e|o)\s*(?:of\s*)?([\d.,]+)\s*mm',
        r'amplitude max\s*([\d.,]+)\s*mm',
        r'max amplitude\s*([\d.,]+)\s*mm',
        r'with\s*([\d.,]+)\s*mm amplitude',
        r'([\d.,]+)\s*mm amplitude',
    ]

    for pattern in patterns:
        for match in re.findall(pattern, text, flags=re.IGNORECASE):
            value = parse_number(match)
            if not math.isnan(value):
                matches.append(value)

    return max(matches) if matches else math.nan

def extract_max_time_seconds(row):
    text = combine_text(row, ['Remarks', 'Description', 'Volcanic_Activity_Summary'])
    values = []

    patterns = [
        r'maximum duration\s*([\d.,\s]+?)\s*(?:second|seconds|s)\b',
        r'duration\s*([\d.,\s]+?)\s*(?:second|seconds|s)\b',
        r'lasted for\s*([\d.,]+)\s*(?:second|seconds|s)\b',
    ]

    for pattern in patterns:
        for match in re.findall(pattern, text, flags=re.IGNORECASE):
            tokens = re.findall(r'\d+(?:[.,]\d+)?', match)
            for token in tokens:
                value = parse_number(token)
                if not math.isnan(value):
                    values.append(value)

    return max(values) if values else math.nan

def extract_distance_km(row):
    text = combine_text(row, ['Remarks'])
    patterns = [
        r'run out distance(?: pdc)?\s*([\d.,]+)\s*km',
        r'approximately\s*([\d.,]+)\s*km',
        r'as far as\s*([\d.,]+)\s*km',
    ]

    values = []
    for pattern in patterns:
        for match in re.findall(pattern, text, flags=re.IGNORECASE):
            value = parse_number(match)
            if not math.isnan(value):
                values.append(value)

    return max(values) if values else math.nan

def extract_timestamp(row):
    if not is_missing(row.get('Timestamp')):
        return row.get('Timestamp')
    return row.get('Issued_Time')

def extract_record(row):
    return {
        'timestamp': extract_timestamp(row),
        'tinggi_letusan_m': extract_height_m(row),
        'kec_angin_km_jam': extract_wind_speed_kmh(row),
        'arah_angin_deg': extract_wind_direction_deg(row),
        'amplitudo': extract_amplitude_mm(row),
        'max_time': extract_max_time_seconds(row),
        'jarak_km': extract_distance_km(row),
        'luas_km2': pd.NA,
        'sudut_deg': pd.NA,
        'radius_km': pd.NA,
    }

In [3]:
extracted_df = pd.DataFrame(source_df.apply(extract_record, axis=1).tolist())
extracted_df['alert_level'] = source_df['Alert_Level'].values
output_columns = ['timestamp', 'alert_level', *template_columns]

for column in template_columns:
    extracted_df[column] = pd.to_numeric(extracted_df[column], errors='coerce')

extracted_df = extracted_df.loc[extracted_df['tinggi_letusan_m'].notna(), output_columns].reset_index(drop=True)

extracted_df.to_csv(OUTPUT_CSV, index=False)

coverage = extracted_df[template_columns].notna().sum().to_frame('filled_rows')
if extracted_df.empty:
    coverage['filled_ratio'] = 0.0
else:
    coverage['filled_ratio'] = (coverage['filled_rows'] / len(extracted_df)).round(3)

print(f'Dataset tersimpan ke: {OUTPUT_CSV}')
print(f'Jumlah baris dengan tinggi letusan: {len(extracted_df)}')
coverage

Dataset tersimpan ke: d:\Projects\volcanic_ash\data\filtered_semeru.csv
Jumlah baris dengan tinggi letusan: 1570


,filled_rows,filled_ratio
tinggi_letusan_m,1570,1.000
kec_angin_km_jam,0,0.000
arah_angin_deg,1570,1.000
amplitudo,1568,0.999
max_time,1568,0.999
jarak_km,0,0.000
luas_km2,0,0.000
sudut_deg,0,0.000
radius_km,0,0.000
